# Criteria Parser Development

Iterative LLM prompt development for structured criteria extraction.

In [ ]:
import sys
sys.path.insert(0, '../src')

from trial_matcher.config import get_settings
from trial_matcher.criteria_parser.extractor import CriteriaExtractor
from trial_matcher.criteria_parser.prompts import build_extraction_prompt, SYSTEM_PROMPT

settings = get_settings()
print(f'Ollama model: {settings.ollama_model}')

In [ ]:
# Test segmentation (no LLM needed)
extractor = CriteriaExtractor(settings)

sample_text = """
Inclusion Criteria:
- HbA1c between 6.5% and 9.5% within the last 3 months
- Age 30 to 75 years
- Type 2 diabetes mellitus diagnosis
- Patient able to make own informed decisions

Exclusion Criteria:
- History of drug abuse or substance use disorder
- Creatinine > 1.5 mg/dL at screening
- Use of dietary supplements within 2 months prior to enrollment
"""

incl, excl = extractor._split_sections(sample_text)
print('INCLUSION:')
for line in incl:
    print(f'  - {line}')
print('\nEXCLUSION:')
for line in excl:
    print(f'  - {line}')

In [ ]:
# Full extraction with Ollama (requires local Ollama)
criteria_set = extractor.extract(sample_text, nct_id='NCT_TEST')
print(f'Parsed {len(criteria_set.inclusion)} inclusion, {len(criteria_set.exclusion)} exclusion criteria')

for c in criteria_set.inclusion:
    print(f'  [{c.type}] {getattr(c, "lab_name", getattr(c, "condition_name", getattr(c, "drug_name", c.raw_text[:50])))}')  

In [ ]:
# Inspect a specific extraction
if criteria_set.inclusion:
    c = criteria_set.inclusion[0]
    print(c.model_dump_json(indent=2))